## Wielowymiarowe ujęcie zróżnicowania poziomu jakości życia w województwach w Polsce
### Projekt Zaliczeniowy - Statystyczna Analiza Danych

W pracy będziemy badać poziom jakości życia w polskich województwach. Pod uwage zostały wzięte zmienne reprezentujące dane sfery życia: samorząd, rynek pracy, warunki mieszkaniowe, społeczeńswto, bezpieczeństwo, środowisko, sport i rekreacja, przedsiebiorczość, i warunki materialne.

Analiza jest inspirowana publikacją naukową pt.: "Wielowymiarowe ujęcie zróżnicowania poziomu jakości życia w województwach w Polsce" autorstwa Elżbiety Roszko-Wójtowicz i Marii M. Grzelak z Uniwersytetu Łódzkiego.

Celem pracy jest zbadanie województw pod kątem atrakcyjności dla przecętnego mieszkańca, wyodrębnienie najlepszych i najgorszych obszarów pod tym kątem, porównanie wyników z wyżej wymienioną analizą.

### 0. Przygotowanie danych

In [ ]:
import os
import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.2f}'.format)

catalog = "./data"
csv_files = []
var_name_list = []

with os.scandir(catalog) as entries:
    for entry in entries:
        if entry.is_file() and entry.name.endswith(".csv"):
            csv_files.append(entry.path)

df_2015 = None
df_2024 = None

for file in csv_files:
    var_name = os.path.basename(file).replace(".csv", "")
    var_name_list.append(var_name)
    df = pd.read_csv(file, sep=";")

    column_2015 = [c for c in df.columns if '2015' in c][0]
    column_2024 = [c for c in df.columns if '2024' in c][0]
    df[column_2015] = pd.to_numeric(df[column_2015].astype(str).str.replace(',', '.').str.strip(), errors='coerce')
    df[column_2024] = pd.to_numeric(df[column_2024].astype(str).str.replace(',', '.').str.strip(), errors='coerce')

    temp_2015 = pd.DataFrame()
    temp_2024 = pd.DataFrame()

    temp_2015['Nazwa'] = df['Nazwa'].copy()
    temp_2015[var_name] = df[column_2015].copy()

    temp_2024['Nazwa'] = df['Nazwa'].copy()
    temp_2024[var_name] = df[column_2024].copy()

    if df_2015 is None:
        df_2015 = temp_2015
        df_2024 = temp_2024
    else:
        df_2015 = pd.merge(df_2015, temp_2015, on='Nazwa')
        df_2024 = pd.merge(df_2024, temp_2024, on='Nazwa')

df_2015 = df_2015.set_index('Nazwa')
df_2024 = df_2024.set_index('Nazwa')

                     emisja_pylkow  gestosc_zaludnienia  naklady_inwestycyjne  \
Nazwa                                                                           
DOLNOŚLĄSKIE                  0.14               145.60                  7800   
KUJAWSKO-POMORSKIE            0.15               116.10                  6692   
LUBELSKIE                     0.08                85.20                  4837   
LUBUSKIE                      0.06                72.80                  5762   
ŁÓDZKIE                       0.15               136.90                  6980   
MAŁOPOLSKIE                   0.19               222.10                  6277   
MAZOWIECKIE                   0.11               150.40                 10868   
OPOLSKIE                      0.16               105.80                  8062   
PODKARPACKIE                  0.08               119.20                  5172   
PODLASKIE                     0.05                58.90                  5754   
POMORSKIE                   

Wynik: dane są przygotowane w osobnych ramkach danych z względu na aktualność danych. W publikacji która stanowi nasz odnośnik wykorzystywany jest: "Wskaźnik wykrywalności przestepstw kryminalnych". My wzieliśmy pod uwagę przestepców, ponieważ GUS nie udostepnia już wskaźnika o takiej nazwie.

### 1. Wprowadzenie - dane

$X1$ - Liczba zarejestrowanych podmiotów gospodarczych w przeliczeniu na 1000 mieszkańców **[podmioty_gospodarcze]**

$X2$ - Udział liczby bezrobotnych w liczbie osób w wieku produkcyjnym (%) **[stopa_bezrobocia]**

$X3$ - Powierzchnia mieszkań oddanych do użytku w m2 na jednego mieszkańca (m2/os.) **[powierzchnia_mieszkaniowa]**

$X4$ - Gęstość zaludnienia (os./km2) **[gestosc_zaludnienia]**

$X5$ - Wskaźnik wykrywalności sprawców przestępstw kryminalnych (%) **[przestepstwa]**

$X6$ - Emisja zanieczyszczeń pyłowych na 1 km2 **[emisja_pylow]**

$X7$ - Udział parków, zieleńców i terenów zieleni osiedlowej w powierzchni ogółem (%) **[udzial_zieleni_w_powierzchni]**

$X8$ - Nakłady inwestycyjne na jednego mieszkańca (zł) **[naklady_inwestycyjne]**

$X9$ - Średnie wynagrodzenie brutto (zł) **[średnie_wynagrodzenie]**

---

### 2. Statystyka Opisowa

#### 2.1 Dodanie funkcji liczącej wszystkie statystyki

Podstawowe miary statystyki opisowej dla każdej zmiennej diagnostycznej:

$\bar{x}$ – średnia arytmetyczna

$M_e$ – mediana

$S_x$ – odchylenie standardowe z próby

$V_s$ – klasyczny współczynnik zmienności

$K$ – kurtoza

$A_s$ – współczynnik asymetrii / skośność

$MIN$ – wartość minimalna wraz z nazwą województwa

$MAX$ – wartość maksymalna wraz z nazwą województwa

Nie będziemy sprawdzać czy istnieją braki w danych (możemy od razu zauważyć, że takowe nie występują), oraz rozważać wartości odstających ze względu na charakter danych.

In [4]:
def oblicz_statystyki_opisowe(df, rok):
    stat = []
    for col in df.columns:
        s = df[col]
        stat.append({
            'Zmienna': col,
            'Rok': rok,
            'Średnia': s.mean(),
            'Mediana': s.median(),
            'Odchylenie standardowe': s.std(ddof=1),
            'Wspołczynnik zmienności': s.std(ddof=1)/s.mean() if s.mean()!=0 else 0,
            'Kurtoza': s.kurtosis(),
            'Skośność': s.skew(),
            'Max': f"{s.idxmax()}: {s.max()}",
            'Min': f"{s.idxmin()}: {s.min()}"
        })
    return pd.DataFrame(stat)

tabela1 = oblicz_statystyki_opisowe(df_2015, '2015')
tabela2 = oblicz_statystyki_opisowe(df_2024, '2024')

tabela = pd.concat([tabela1, tabela2], ignore_index=True)
tabela = tabela.sort_values(by=['Zmienna', 'Rok']).reset_index(drop=True)
tabela = tabela.set_index(['Zmienna', 'Rok'])

#### 2.2 Analiza statystyk opisowych

In [16]:
display(tabela.loc[[var_name_list[0]]])

Średnia  Mediana  Odchylenie standardowe  \
Zmienna       Rok                                              
emisja_pylkow 2015     0.16     0.12                    0.18   
              2024     0.06     0.04                    0.05   

                    Wspołczynnik zmienności  Kurtoza  Skośność            Max  \
Zmienna       Rok                                                               
emisja_pylkow 2015                     1.13    14.57      3.70  ŚLĄSKIE: 0.83   
              2024                     0.95    11.65      3.19  ŚLĄSKIE: 0.25   

                                          Min  
Zmienna       Rok                              
emisja_pylkow 2015  WARMIŃSKO-MAZURSKIE: 0.04  
              2024  WARMIŃSKO-MAZURSKIE: 0.01

Patrząc na średnią czy medianę dla obu lat, wartość tego wskaźnika jest niska. Widzimy spadek o 0.1 w średniej i 0.08 w medianie. Emisja pyłków dla poszczegółnych województw mocno sie waha. Możemy to stwierdzić po dużej wartości odchylenia standardowego, jak i różnicy między maksymalną i minimalną wartością. Jeśli chodzi o województwa dla których te wyniki są osiągane, śląskie i warmińsko-mazurskie, to te obszary nie stanowią zaskoczenia. Wpływa na to przemysł, gęste zaludnie, ogrzewanie węglowe i ukształtowania terenu na Śląsku, oraz jeśli chodzi o Mazury - duża ilość jezior i zalesienie sprzyjają czystszemu powietrzu. Współczynnik zmienności mówi nam również o ogromnym zróznicowaniu województw pod kątem zanieczyszczeń pyłowych (ponad 100% w 2015, w 2024 ta wartośc zmalała, nadal jednak sięga prawie 100%). Obserwujemy także skośność prawostronną. Oznacza to, że w większości województw emisja pyłków jest na poziomie ponieżej średniej. Widzimy także, że często występują wartości odstające (reprezentuje je np. woj. Śląskie), ale również duża część odczytów koncentruję się wokół średniej. Na przestrzeni tych 9 lat możemy także obserwować zmiejszenie się dysproporcji zanieczyszcenia powietrza województwami między. Wynika to prawdopodnie z rosnącej świadomości ekologicznej jak i idących za tym przepisów prawnych.

In [17]:
display(tabela.loc[[var_name_list[1]]])

Średnia  Mediana  Odchylenie standardowe  \
Zmienna             Rok                                              
gestosc_zaludnienia 2015   126.03   116.10                   74.93   
                    2024   121.45   110.40                   72.13   

                          Wspołczynnik zmienności  Kurtoza  Skośność  \
Zmienna             Rok                                                
gestosc_zaludnienia 2015                     0.59     7.12      2.43   
                    2024                     0.59     5.80      2.22   

                                     Max                        Min  
Zmienna             Rok                                              
gestosc_zaludnienia 2015  ŚLĄSKIE: 370.6            PODLASKIE: 58.9  
                    2024  ŚLĄSKIE: 347.9  WARMIŃSKO-MAZURSKIE: 55.8

Widzimy, ze wartość tego wskaźnika osiąga warości umiarkowane, jeśli patrzymy na średnią czy medianę. Gęstość zaludnienia mocno się jednak waha między województwami (odchylenie standardowe stanowy przeszło połowę średniej, wysoki współczynnik zmienności, który nie zmienił się na przestrzeni lat). Po raz kolejny mamy tu do czynienia ze skośnoscią prawostronną. Kurtoza osiąga niższe warości dla analizowanego wskaźnika, lecz nadal jest wysoka, zwłaszcza w  2015. Świadczy to o grubych ogonach i istnieniu outlinerów. Przykładem takiego outlinera jest województwo Śląskie z ok. 3 razy większą gęstością zaludnienia w porównaniu do średniej. Naprzeciw wychodzi woj. podlaskie (2015) i woj. warmińsko-mazurskie (2024) z wynikiem o około połowę mniejszym od średniej. Oba te przypadki wpływają na wartość kurtozy i skośności. Te różnice wyniką m.in z wielkości województw, urbanizacji, ukształtowania terenu.

In [18]:
display(tabela.loc[[var_name_list[2]]])

Średnia  Mediana  Odchylenie standardowe  \
Zmienna              Rok                                              
naklady_inwestycyjne 2015  6599.65  6383.00                 1470.56   
                     2024 10780.82 10553.00                 2939.86   

                           Wspołczynnik zmienności  Kurtoza  Skośność  \
Zmienna              Rok                                                
naklady_inwestycyjne 2015                     0.22     3.55      1.45   
                     2024                     0.27     9.43      2.70   

                                          Max                        Min  
Zmienna              Rok                                                  
naklady_inwestycyjne 2015  MAZOWIECKIE: 10868       ŚWIĘTOKRZYSKIE: 4624  
                     2024  MAZOWIECKIE: 20911  WARMIŃSKO-MAZURSKIE: 7741

Jeśli chodzi o nakłady inwestycyjne na 1 mieszkańca średnio prawie 7000 zł (2015) i 11 000zł (2024), to dobry wynik. 
Rok 2015 jest rokiem w którym możemy zaobserwować mniejsze dysproporcje niż w 2024. Współczynnik zmienności mówi nam o małej lub umiarkowanej zmienności, lecz widzimy wzrost wartości kurtozy i skośności w rozważanych okresach. Potwierdza to również różnica między wartością maksymalną i minimalną (w 2015 około 6 tys., w 2024 około 13 tys.). Województwo mazowieckie utrzymało swoją pozycję jako lidera w tej kategorii (co jest naturalną konsekwencją koncentracji kapitału w regionie stołecznym).

In [19]:
display(tabela.loc[[var_name_list[3]]])

Średnia  Mediana  Odchylenie standardowe  \
Zmienna              Rok                                              
podmioty_gospodarcze 2015   105.35   102.00                   19.41   
                     2024   134.76   127.00                   24.46   

                           Wspołczynnik zmienności  Kurtoza  Skośność  \
Zmienna              Rok                                                
podmioty_gospodarcze 2015                     0.18    -0.96      0.31   
                     2024                     0.18    -0.18      0.64   

                                        Max                Min  
Zmienna              Rok                                        
podmioty_gospodarcze 2015  MAZOWIECKIE: 143   PODKARPACKIE: 78  
                     2024  MAZOWIECKIE: 191  PODKARPACKIE: 104

Zarejestrowane podmioty gospodarcze w przeliczeniu na 1000 mieszkańców reprezentują w naszej analizie stan przedsiębiorczości Polaków. Patrząc na tabele możemy dostrzec, że kształtuje się na umiarkowanym poziomie. Niskie odchylenie standarodowe, współczynnik zmienności, mała różnica między maksumum i minimum oraz ujemna kurtoza wskazują na spłaszczony rozkład wyników i brak drastycznych wartości odstających między regionami. Po raz kolejny najwększy wynik osiąga mazowieckie, na ostatnim miejscu plasuje się podkarpackie.

In [20]:
display(tabela.loc[[var_name_list[4]]])

Średnia  Mediana  Odchylenie standardowe  \
Zmienna                   Rok                                              
powierzchnia_mieszkaniowa 2015    26.71    26.50                    1.49   
                          2024    31.80    31.90                    1.51   

                                Wspołczynnik zmienności  Kurtoza  Skośność  \
Zmienna                   Rok                                                
powierzchnia_mieszkaniowa 2015                     0.06    -0.28     -0.14   
                          2024                     0.05     0.10     -0.42   

                                              Max                        Min  
Zmienna                   Rok                                                 
powierzchnia_mieszkaniowa 2015  MAZOWIECKIE: 29.5  WARMIŃSKO-MAZURSKIE: 23.9  
                          2024  MAZOWIECKIE: 34.5  WARMIŃSKO-MAZURSKIE: 28.6

Jeśli chodzi o Polskę, średnia powierzchnia mieszkaniowa wynosząca blisko 30 $m^2$, to przeciętny wynik. W porówaniu do standardu europejskiego (35 - 40 $m^2$) wynik ten klasuję się poniżej średniej. Po wartości pozostałych wskażników, możemy stwierdzić, że nie ma istotnych różnic między województwami. Mamy małe odchylenie standarowe, niski współczynnik zmieności, mało outlinerów, wartości naszego wskażnika są raczej skoncentrowane, z tendencją bycia wyższymi od średniej. Biorąc pod uwage maksymalną i minimalną wartość, ona województwa zachowały swoje pozycje.

In [21]:
display(tabela.loc[[var_name_list[5]]])

Średnia  Mediana  Odchylenie standardowe  \
Zmienna      Rok                                              
przestepstwa 2015    66.48    67.10                    5.73   
             2024    70.37    68.40                    4.97   

                   Wspołczynnik zmienności  Kurtoza  Skośność  \
Zmienna      Rok                                                
przestepstwa 2015                     0.09     0.18     -0.63   
             2024                     0.07    -0.99     -0.09   

                                    Max                Min  
Zmienna      Rok                                            
przestepstwa 2015  ŚWIĘTOKRZYSKIE: 76.4  MAZOWIECKIE: 53.8  
             2024     MAŁOPOLSKIE: 78.0  MAZOWIECKIE: 61.8

Wkaźnik wykrywalności sprawców przestępstw kryminalnych wynoszący średnio około 70% w obu latach to dobry wynik. 
Nie mamy tu do czynienia z wysokim zróżnicowaniem między województwami, mało wartości odstających, są one raczej skoncentrowane. Widzimy tendencję wartości wskaźnika do bycia zbliżonym do średniej w wielu województwach. Lekka skośność lewostronna, ale rozkła jest zbliżony do normalnego w obu latach (zwłaszcza w 2024). Najbezpiecznijsze województwa to świętokrzystkie i małopolskie. Po drugiej stronie znajduje się mazowieckie.

In [22]:
display(tabela.loc[[var_name_list[6]]])

Średnia  Mediana  Odchylenie standardowe  \
Zmienna               Rok                                              
srednie_wynagrodzenie 2015  3853.89  3790.76                  393.38   
                      2024  8156.40  7958.35                  640.67   

                            Wspołczynnik zmienności  Kurtoza  Skośność  \
Zmienna               Rok                                                
srednie_wynagrodzenie 2015                     0.10     5.82      2.18   
                      2024                     0.08     3.39      1.71   

                                              Max  \
Zmienna               Rok                           
srednie_wynagrodzenie 2015   MAZOWIECKIE: 5098.55   
                      2024  MAZOWIECKIE: 10018.71   

                                                     Min  
Zmienna               Rok                                 
srednie_wynagrodzenie 2015  WARMIŃSKO-MAZURSKIE: 3495.02  
                      2024         PODKARPACKIE: 7514.69

Średnie wynagrodzenie brutto wzrosło w ciągu 9 lat ponad dwukrotnie, co ma związek m.in. z inflacją oraz wzrostem kosztów utrzymania. Zróżnicowanie między województwami w ramach tego wskaźnika jest stosunkowo małe. Występują jednak wartości odstające, ltóre wpływają na wynik kurtozy w 2015 roku. W tym okresie możemy także zaobserwować większą skośność prwostronną niż w 2024 roku. Możemy z tego wywnioskować, że pomimo ogólnego wzrostu płac i kosztów życia, dysproporcje płacowe między województwami powoli maleją.

In [23]:
display(tabela.loc[[var_name_list[7]]])

Średnia  Mediana  Odchylenie standardowe  \
Zmienna          Rok                                              
stopa_bezrobocia 2015    10.82    10.50                    2.59   
                 2024     3.89     4.00                    0.96   

                       Wspołczynnik zmienności  Kurtoza  Skośność  \
Zmienna          Rok                                                
stopa_bezrobocia 2015                     0.24    -0.34      0.15   
                 2024                     0.25    -1.18     -0.08   

                                             Max                 Min  
Zmienna          Rok                                                  
stopa_bezrobocia 2015  WARMIŃSKO-MAZURSKIE: 16.2  WIELKOPOLSKIE: 6.1  
                 2024          PODKARPACKIE: 5.5  WIELKOPOLSKIE: 2.3

Widzimy, że stopa bezrobocia zmalała prawie trzykrotnie w okresie 2015 - 2024. Odchylenie standardowe tego wskaźnika także zmalało, choć współczynnik zmienności nieznacznie wzrósł. Wynosi on jednak 24 - 25%, co wciąż pozwala uznać populację za stosunkowo jednorodną pod tym względem. Zmiana wartości skośności pokazuje ewolucję rozkładu względem średniej (w 2015 r. występujeła lekka skośność prawostronna, w 2024 r. rozkład jest bardziej symetryczny). Ponieważ wartości skośności w obu latach są bliskie zera, układ można uznać za zbliżony do normalnego.

In [24]:
display(tabela.loc[[var_name_list[8]]])

Średnia  Mediana  Odchylenie standardowe  \
Zmienna                      Rok                                              
udzial_zieleni_w_powierzchni 2015     0.20     0.20                    0.17   
                             2024     0.22     0.20                    0.14   

                                   Wspołczynnik zmienności  Kurtoza  Skośność  \
Zmienna                      Rok                                                
udzial_zieleni_w_powierzchni 2015                     0.83    12.05      3.26   
                             2024                     0.63    10.15      2.88   

                                            Max             Min  
Zmienna                      Rok                                 
udzial_zieleni_w_powierzchni 2015  ŚLĄSKIE: 0.8  LUBELSKIE: 0.1  
                             2024  ŚLĄSKIE: 0.7  LUBELSKIE: 0.1

Średni udział zieleni w powierzchni ogółem nieznacznie się zmienił w ciągu analizowanyh lat. Stabilność tego wskaźnika świadczy o stałym utrzymywaniu miejskiej infrastruktury zielonej w regionach. Najwyższą wartość wskaźnika odnotowujemy w województwie śląskim (0,8% w 2015 r. oraz 0,7% w 2024 r.). Wynika to z faktu, że Śląsk jest regionem o najwyższym stopniu urbanizacji i zagęszczenia miast, co wiąże się z zagęszczeniem osiedli oraz rozbudowaną siecią parków miejskich i zieleńców. Najmniejsza wartość przypada niezmiennie lubelskiemu, co wiąże się z jego silnie rolniczym charakterem (duży udział gruntów ornych kosztem zwartych kompleksów leśnych). Na uwagę zasługuje wysoka kurtoza, świadcząca o obecności wartości odstających, oraz znaczna niejednorodność populacji (współczynnik zmienności na poziomie 63–83%).

### 4. Analiza skupień


#### 4.1 Grupowanie podziałowe - metoda k-medoid

Dzięki powyższej analizie możemy zauważyć, że istnieją województwa, które można określić jako wartości odstające. Z tego względu zastosujemy metodę k-medoid

In [12]:
from sklearn.preprocessing import StandardScaler
from sklearn_extra.cluster import KMedoids
from sklearn.metrics import silhouette_score

def k_medoid (df):
    df_scaled = StandardScaler().fit_transform(df)
    sil_core = -1
    k = 0
    for i in range(2,6):
        kmedoids = KMedoids(n_clusters=i, random_state=42, metric='euclidean')
        labels = kmedoids.fit_predict(df_scaled)
        if len(set(labels)) < i:
            continue
        score = silhouette_score(df_scaled, labels) #metoda sylwetki
        if (score > sil_core):
            sil_core = score
            k = i
    kmedoids = KMedoids(n_clusters=k, random_state=42, metric='euclidean')
    df['Skupienie'] = kmedoids.fit_predict(df_scaled)


k_medoid(df_2015)
tabela_medoid_2015 = df_2015.groupby('Skupienie').mean()
display(tabela_medoid_2015)
grupy_wojewodztw_2015 = df_2015.groupby('Skupienie').apply(lambda x: list(x.index)).reset_index(name='Województwa')
for idx, row in grupy_wojewodztw_2015.iterrows():
    print(f"Skupienie {row['Skupienie']}: {', '.join(row['Województwa'])}")



c:\Users\Wiktoria\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn_extra\cluster\_k_medoids.py:329: UserWarning: Cluster 2 is empty! self.labels_[self.medoid_indices_[2]] may not be labeled with its corresponding cluster (2).
  warnings.warn(
c:\Users\Wiktoria\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn_extra\cluster\_k_medoids.py:329: UserWarning: Cluster 2 is empty! self.labels_[self.medoid_indices_[2]] may not be labeled with its corresponding cluster (2).
  warnings.warn(
c:\Users\Wiktoria\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn_extra\cluster\_k_medoids.py:329: UserWarning: Cluster 2 is empty! self.labels_[self.medoid_indices_[2]] may not be labeled with its corresponding cluster (2).
  warnings.warn(


,emisja_pylkow,gestosc_zaludnienia,naklady_inwestycyjne,podmioty_gospodarcze,powierzchnia_mieszkaniowa,przestepstwa,srednie_wynagrodzenie,stopa_bezrobocia,udzial_zieleni_w_powierzchni
Skupienie,,,,,,,,,
0,0.10,88.51,5848.50,99.50,26.09,69.68,3636.87,12.14,0.13
1,0.24,179.63,7672.71,113.71,27.59,61.90,4163.91,8.94,0.30


Skupienie 0: KUJAWSKO-POMORSKIE, LUBELSKIE, LUBUSKIE, PODKARPACKIE, PODLASKIE, ŚWIĘTOKRZYSKIE, WARMIŃSKO-MAZURSKIE, WIELKOPOLSKIE, ZACHODNIOPOMORSKIE, ZACHODNIOPOMORSKIE
Skupienie 1: DOLNOŚLĄSKIE, ŁÓDZKIE, MAŁOPOLSKIE, MAZOWIECKIE, OPOLSKIE, POMORSKIE, ŚLĄSKIE


C:\Users\Wiktoria\AppData\Local\Temp\ipykernel_17424\95354273.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grupy_wojewodztw_2015 = df_2015.groupby('Skupienie').apply(lambda x: list(x.index)).reset_index(name='Województwa')
